In [8]:
import pandas as pd
import numpy as np

In [9]:
df_train = pd.read_csv("../clean_data/train_data.csv")

# Preprocess train data
if 'publish_time_utc' in df_train.columns:
    df_train.drop(columns=['publish_time_local', 'publish_time_utc'], inplace=True)

# Add 'solar_' prefix to columns by position 3 through 8 (inclusive)
start_idx, end_idx = 2, 7
cols = list(df_train.columns)
for i in range(start_idx, end_idx + 1):
    if i < len(cols):
        col = cols[i]
        if not col.startswith('solar_'):
            cols[i] = 'solar_' + col
df_train.columns = cols

if 'solar_gen_system_wide_x' in df_train.columns:    # Remove trailing '_x' if present
    df_train.rename(columns={'solar_gen_system_wide_x': 'solar_gen_system_wide'}, inplace=True)

# Add 'wind_' prefix to columns by position 9 through 12 (inclusive)
start_idx, end_idx = 8, 11
cols = list(df_train.columns)
for i in range(start_idx, end_idx + 1):
    if i < len(cols):
        col = cols[i]
        if not col.startswith('wind_'):
            cols[i] = 'wind_' + col
df_train.columns = cols

if 'wind_gen_system_wide_y' in df_train.columns:   # Remove trailing '_y' if present
    df_train.rename(columns={'wind_gen_system_wide_y': 'wind_gen_system_wide'}, inplace=True)

# Remove weather columns
weather_cols = ['TEMP_qc' ,'DEW_qc', 'SLP_qc', 'WIND_DIR_deg', 'WIND_DIR_qc', 'WIND_SPD_qc']
df_train.drop(columns=weather_cols, inplace=True)

# Choose hub to predict
hubs = ['HB_BUSAVG', 'HB_HOUSTON', 'HB_HUBAVG', 'HB_NORTH', 'HB_PAN', 'HB_SOUTH', 'HB_WEST']
hub_to_predict = 'HB_NORTH'

# Keep only the chosen hub column in df_train
#  - Drops other hub columns listed in `hubs`.
existing_hubs_train = [h for h in hubs if h in df_train.columns]
# Drop other hubs from df_train
drop_hubs_train = [h for h in existing_hubs_train if h != hub_to_predict]
if drop_hubs_train:
    df_train.drop(columns=drop_hubs_train, inplace=True)

# Move target column to the end
target_col = hub_to_predict
if target_col in df_train.columns:
    cols = [col for col in df_train.columns if col != target_col] + [target_col]
    df_train = df_train[cols]

# Fill NaNs with last valid observation forward
df_train.fillna(method='ffill', inplace=True)

# Set index to 'interval_start_local'
df_train.set_index('interval_start_local', inplace=True)
df_train.index = pd.to_datetime(df_train.index)

# Show result
df_train


/var/folders/0r/712pw3x90ngc941898p_pfg80000gn/T/ipykernel_4342/2755787623.py:56: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_train.fillna(method='ffill', inplace=True)


,load,solar_gen_system_wide,solar_gen_centerwest,solar_gen_northwest,solar_gen_fareast,solar_gen_southeast,solar_gen_centereast,wind_gen_system_wide,wind_gen_lz_south_houston,wind_gen_lz_west,wind_gen_lz_north,TEMP_C,DEW_C,SLP_hPa,WIND_SPD_ms,HB_NORTH
interval_start_local,,,,,,,,,,,,,,,,
2023-01-01 00:00:00,34969.250000,0.45,0.01,0.00,0.36,0.00,0.07,21752.91,4318.38,15100.73,2333.80,17.75,6.70,1007.75,4.1,-2.56
2023-01-01 01:00:00,35573.500000,0.46,0.01,0.00,0.37,0.00,0.07,21569.51,3685.25,15368.05,2516.21,16.70,7.20,1007.90,4.6,-1.51
2023-01-01 02:00:00,36279.750000,0.45,0.01,0.00,0.36,0.00,0.07,21035.48,3544.24,14954.69,2536.55,16.10,7.20,1008.70,3.1,-1.01
2023-01-01 03:00:00,36765.833333,0.46,0.01,0.00,0.37,0.00,0.07,20595.37,3571.74,14505.72,2517.91,15.85,7.20,1008.80,3.6,-0.07
2023-01-01 04:00:00,37049.916667,0.45,0.01,0.00,0.36,0.00,0.07,20387.59,3295.90,14590.41,2501.28,15.00,7.20,1009.40,4.6,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-30 13:00:00,49497.333333,10308.84,470.95,962.55,4496.30,1444.14,1412.20,21834.62,2976.84,16302.05,2555.73,11.10,7.20,1005.10,5.7,-3.21
2024-12-30 14:00:00,48549.583333,12919.12,1141.19,1005.58,5351.85,1986.38,1583.28,19823.11,2560.70,14585.54,2676.87,12.20,7.20,1004.50,8.2,-2.73
2024-12-30 15:00:00,47718.000000,13880.78,1272.04,1031.21,5727.15,2126.71,1648.66,17393.12,2072.08,12857.21,2463.83,13.60,6.95,1004.35,7.7,-2.18


In [10]:
df_test = pd.read_csv("../clean_data/test_data_baseline.csv")
# Preprocess test data
date_cols = ['interval_end_local', 'publish_time_local', 'publish_time_utc', 'interval_start_utc', 'interval_end_utc']
load_cols = ['coast', 'east', 'far_west', 'north', 'north_central', 'south_central', 'southern','west']
weather_cols = ['TEMP_qc' ,'DEW_qc', 'SLP_qc', 'WIND_DIR_deg', 'WIND_DIR_qc', 'WIND_SPD_qc']

df_test.drop(columns=date_cols, inplace=True)
df_test.drop(columns=load_cols, inplace=True)
df_test.drop(columns=weather_cols, inplace=True)

df_test.rename(columns={'system_total': 'load'}, inplace=True)

df_test.rename(columns={'gen_system_wide_x': 'gen_system_wide'}, inplace=True)
df_test.rename(columns={'gen_system_wide_y': 'wind_gen_system_wide'}, inplace=True)

# Add 'solar_' prefix to columns by position 3 through 8 (inclusive)
start_idx, end_idx = 2, 7
cols = list(df_test.columns)
for i in range(start_idx, end_idx + 1):
    if i < len(cols):
        col = cols[i]
        if not col.startswith('solar_'):
            cols[i] = 'solar_' + col
df_test.columns = cols

# Add 'wind_' prefix to columns by position 9 through 12 (inclusive)
start_idx, end_idx = 8, 11
cols = list(df_test.columns)
for i in range(start_idx, end_idx + 1):
    if i < len(cols):
        col = cols[i]
        if not col.startswith('wind_'):
            cols[i] = 'wind_' + col
df_test.columns = cols

# Choose hub to predict
hubs = ['HB_BUSAVG', 'HB_HOUSTON', 'HB_HUBAVG', 'HB_NORTH', 'HB_PAN', 'HB_SOUTH', 'HB_WEST']
hub_to_predict = 'HB_NORTH'

# Keep only the chosen hub column in df_test
#  - Drops other hub columns listed in `hubs`.
existing_hubs_test = [h for h in hubs if h in df_test.columns]
# Drop other hubs from df_test
drop_hubs_test = [h for h in existing_hubs_test if h != hub_to_predict]
if drop_hubs_test:
    df_test.drop(columns=drop_hubs_test, inplace=True)

# Remove NaNs at the end of df_test
#df_test.dropna(inplace=True)


last_idx = df_test['TEMP_C'].last_valid_index()
if last_idx is not None:
    last_val = df_test.at[last_idx, 'TEMP_C']
    print("Last non-NaN index:", last_idx)
    print("Last non-NaN value:", last_val)
else:
    print("Column 'TEMP_C' contains only NaNs")
    last_val = np.nan

# Move target column to the end
target_col = hub_to_predict
if target_col in df_test.columns:
    cols = [col for col in df_test.columns if col != target_col] + [target_col]
    df_test = df_test[cols]

# Remove rows after last_idx
if last_idx is not None:
    df_test = df_test.loc[:last_idx]

# Fill NaNs as average of last valid value for each column
df_test.fillna(method='ffill', inplace=True)

# Set index to 'interval_start_local'
df_test.set_index('interval_start_local', inplace=True)
df_test.index = pd.to_datetime(df_test.index)
df_test

Last non-NaN index: 5709
Last non-NaN value: 23.9


/var/folders/0r/712pw3x90ngc941898p_pfg80000gn/T/ipykernel_4342/4246065452.py:72: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_test.fillna(method='ffill', inplace=True)


,load,solar_gen_system_wide,solar_gen_centerwest,solar_gen_northwest,solar_gen_fareast,solar_gen_southeast,solar_gen_centereast,wind_gen_system_wide,wind_gen_lz_south_houston,wind_gen_lz_west,wind_gen_lz_north,TEMP_C,DEW_C,SLP_hPa,WIND_SPD_ms,HB_NORTH
interval_start_local,,,,,,,,,,,,,,,,
2025-01-01 00:00:00,44106.88,0.72,0.00,0.00,0.35,0.00,0.36,14370.12,3832.30,9177.81,1360.01,6.4,1.4,1023.5,4.90,20.69
2025-01-01 01:00:00,43816.13,0.75,0.00,0.00,0.36,0.00,0.37,14889.31,3965.46,9471.44,1452.41,5.6,1.1,1024.4,4.60,22.07
2025-01-01 02:00:00,43633.94,0.73,0.00,0.00,0.35,0.00,0.36,15306.80,4204.55,9689.47,1412.78,5.0,0.6,1025.0,4.10,23.49
2025-01-01 03:00:00,43521.12,0.75,0.00,0.00,0.37,0.00,0.36,15026.01,4371.51,9209.34,1445.16,4.7,0.6,1025.3,4.35,14.78
2025-01-01 04:00:00,43393.86,0.76,0.00,0.00,0.37,0.00,0.36,14531.85,4379.07,8431.45,1721.33,3.9,0.0,1026.0,4.10,8.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-26 19:00:00,73805.70,2758.27,487.64,303.76,418.51,166.13,298.92,12722.90,3818.06,8015.06,889.78,27.2,17.2,1018.5,4.60,27.00
2025-08-26 20:00:00,71613.72,40.96,3.80,8.70,0.38,0.38,0.02,12610.89,3403.30,8258.04,949.55,25.6,17.8,1018.8,4.60,25.57
2025-08-26 21:00:00,68819.28,0.57,0.00,0.00,0.41,0.00,0.15,12786.03,3107.07,8495.09,1183.87,25.0,17.2,1019.4,5.10,25.43


In [11]:
# Feature Engineering
# Lags
lag_amount = [1, 2, 4, 8, 12, 24, 48]
lag_features = ['load', 'gen_syst']

# Define column subsets
# Wind columns

def time_features(df):
    df['day_of_week'] = pd.to_datetime(df['interval_start_local']).dt.dayofweek
    df['weekend'] = df['day_of_week'].apply(lambda x: 1 if x in [5, 6] else 0)

def weather_features(df):
    df['TEMP_F'] = df['TEMP_C'] * 9/5 + 32
    df['degree_days'] = abs(df['TEMP_F'] - 65)


In [12]:
# -----------------------------
# TIME FEATURES
# -----------------------------
def make_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["time_hour"] = df.index.hour
    out["time_dayofweek"] = df.index.dayofweek
    out["time_is_weekend"] = (out["time_dayofweek"] >= 5).astype(int)
    out["time_hour_sin"] = np.sin(2 * np.pi * out["time_hour"] / 24)
    out["time_hour_cos"] = np.cos(2 * np.pi * out["time_hour"] / 24)
    return out


# -----------------------------
# WEATHER FEATURES
# TEMP_C → °F, CDD/HDD, dewpoint, etc.
# Only raw vars are lagged; rolling stats use history only.
# -----------------------------
def make_weather_features(df: pd.DataFrame,
                          lags=(1, 2, 4, 12, 24),
                          temp_col="TEMP_C") -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    temp_f = df[temp_col] * 9/5 + 32
    out["weather_temp_f"] = temp_f
    out["weather_CDD65"] = (temp_f - 65).clip(lower=0)
    out["weather_HDD65"] = (65 - temp_f).clip(lower=0)
    out["weather_dewpoint_C"] = df["DEW_C"]
    out["weather_deltaT_dew_C"] = df["TEMP_C"] - df["DEW_C"]
    out["weather_slp_hPa"] = df["SLP_hPa"]
    out["weather_windspd_ms"] = df["WIND_SPD_ms"]

    # History-only rolling stats (no t info)
    hist_temp_f = temp_f.shift(24)
    out["weather_temp_7d_mean"] = hist_temp_f.rolling(24*7, min_periods=1).mean()
    out["weather_temp_anom_24h"] = hist_temp_f - out["weather_temp_7d_mean"]

    # LAG ONLY RAW-LEVEL WEATHER VARS
    raw_cols = [
        "weather_temp_f",
        "weather_CDD65",
        "weather_HDD65",
        "weather_dewpoint_C",
        "weather_deltaT_dew_C",
        "weather_slp_hPa",
        "weather_windspd_ms",
    ]
    for lag in lags:
        for col in raw_cols:
            out[f"{col}_lag{lag}"] = out[col].shift(lag)

    return out


# -----------------------------
# LOAD FEATURES
# -----------------------------
def make_load_features(df: pd.DataFrame,
                       lags=(1, 2, 4, 12, 24),
                       load_col="load") -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    out["load_system"] = df[load_col]

    # History-only stats (use load up to t-1)
    hist_load = df[load_col].shift(24)
    out["load_ramp_hist"] = hist_load.diff()
    out["load_rolling_6h_hist"] = hist_load.rolling(6, min_periods=1).mean()
    out["load_rolling_24h_hist"] = hist_load.rolling(24, min_periods=1).mean()
    out["load_vol_24h_hist"] = hist_load.rolling(24, min_periods=2).std()

    # LAG ONLY RAW LOAD
    raw_cols = ["load_system"]
    for lag in lags:
        for col in raw_cols:
            out[f"{col}_lag{lag}"] = out[col].shift(lag)

    return out


# -----------------------------
# SOLAR FEATURES (multi-regional)
# -----------------------------
def make_solar_features(df: pd.DataFrame,
                        lags=(1, 2, 4, 12, 24)) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    solar_cols = [c for c in df.columns if c.startswith("solar_gen_")]

    # Levels
    for col in solar_cols:
        suffix = col.replace("solar_gen_", "")
        out[f"solar_{suffix}"] = df[col]

    # Regional shares
    region_cols = [c for c in solar_cols if c != "solar_gen_system_wide"]
    # if region_cols:
    #     total_regions = df[region_cols].sum(axis=1)
    #     out["solar_total_regions"] = total_regions
    #     denom = total_regions.replace(0, np.nan)
    #     for col in region_cols:
    #         suffix = col.replace("solar_gen_", "")
    #         out[f"solar_share_{suffix}"] = df[col] / denom

    # LAG ONLY RAW SOLAR LEVELS (not shares)
    raw_cols = [
        c for c in out.columns
        if c.startswith("solar_") and not c.startswith("solar_share_")
    ]
    for lag in lags:
        for col in raw_cols:
            out[f"{col}_lag{lag}"] = out[col].shift(lag)

    return out


# -----------------------------
# WIND FEATURES (multi-regional)
# -----------------------------
def make_wind_features(df: pd.DataFrame,
                       lags=(1, 2, 4, 12, 24)) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    wind_cols = [c for c in df.columns if c.startswith("wind_gen_")]

    for col in wind_cols:
        suffix = col.replace("wind_gen_", "")
        out[f"wind_{suffix}"] = df[col]

    region_cols = [c for c in wind_cols if c != "wind_gen_system_wide"]
    # if region_cols:
    #     total_regions = df[region_cols].sum(axis=1)
    #     out["wind_total_regions"] = total_regions
    #     denom = total_regions.replace(0, np.nan)
    #     for col in region_cols:
    #         suffix = col.replace("wind_gen_", "")
    #         out[f"wind_share_{suffix}"] = df[col] / denom

    # LAG ONLY RAW WIND LEVELS (not shares)
    raw_cols = [
        c for c in out.columns
        if c.startswith("wind_") and not c.startswith("wind_share_")
    ]
    for lag in lags:
        for col in raw_cols:
            out[f"{col}_lag{lag}"] = out[col].shift(lag)

    return out


# -----------------------------
# PRICE FEATURES (HB_NORTH)
# -----------------------------
def make_price_features(df: pd.DataFrame,
                        lags=(1, 2, 4, 12, 24),
                        price_col="HB_NORTH") -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    out["price_HB_NORTH"] = df[price_col]

    hist_price = df[price_col].shift(24)
    out["price_ramp_hist"] = hist_price.diff()
    out["price_rolling_6h_hist"] = hist_price.rolling(6, min_periods=1).mean()
    out["price_rolling_24h_hist"] = hist_price.rolling(24, min_periods=1).mean()
    out["price_vol_24h_hist"] = hist_price.rolling(24, min_periods=2).std()

    # LAG ONLY RAW PRICE
    for lag in lags:
        out[f"price_lag{lag}"] = df[price_col].shift(lag)

    return out


# -----------------------------
# DROP RAW ACTUALS (NO NON-LAGGED REALIZED VALUES)
# -----------------------------
def drop_raw_actuals(df: pd.DataFrame) -> pd.DataFrame:
    to_drop = []

    for col in df.columns:
        # Original raw columns from your CSV
        if col in ["load", "TEMP_C", "DEW_C", "SLP_hPa", "WIND_SPD_ms"]:
            to_drop.append(col)
        if col.startswith("solar_gen_") or col.startswith("wind_gen_"):
            to_drop.append(col)

        # Engineered base-level actuals (keep only their lags / history stats)
        if col in [
            "weather_temp_f", "weather_CDD65", "weather_HDD65",
            "weather_dewpoint_C", "weather_deltaT_dew_C",
            "weather_slp_hPa", "weather_windspd_ms",
            "load_system",
            "solar_system_wide", "solar_centerwest", "solar_northwest",
            "solar_fareast", "solar_southeast", "solar_centereast",
            "solar_total_regions",
            "wind_system_wide", "wind_lz_south_houston",
            "wind_lz_west", "wind_lz_north", "wind_total_regions",
            "price_HB_NORTH",
        ]:
            to_drop.append(col)

    # dedupe with dict trick
    to_drop = list(dict.fromkeys(to_drop))
    return df.drop(columns=to_drop, errors="ignore")


# -----------------------------
# MAIN PIPELINE
# -----------------------------
def build_feature_set(df: pd.DataFrame,
                      lags=(1, 2, 4, 12, 24)) -> pd.DataFrame:
    df = df.copy()

    # Datetime index from interval_start_local
    if not isinstance(df.index, pd.DatetimeIndex):
        if "interval_start_local" not in df.columns:
            raise ValueError("Expected 'interval_start_local'")
        df["interval_start_local"] = pd.to_datetime(df["interval_start_local"])
        df = df.set_index("interval_start_local").sort_index()

    time_feats   = make_time_features(df)
    weather_feats = make_weather_features(df, lags=lags)
    load_feats   = make_load_features(df, lags=lags)
    solar_feats  = make_solar_features(df, lags=lags)
    wind_feats   = make_wind_features(df, lags=lags)
    price_feats  = make_price_features(df, lags=lags, price_col="HB_NORTH")

    out = pd.concat(
        [
            df[["HB_NORTH"]],   # target
            time_feats,
            weather_feats,
            load_feats,
            solar_feats,
            wind_feats,
            price_feats,
        ],
        axis=1,
    )

    out = drop_raw_actuals(out)

    # Order columns by group: time, weather, load, solar, wind, price
    group_order = ["time_", "weather_", "load_", "solar_", "wind_", "price_"]
    target = ["HB_NORTH"]
    ordered = []

    for prefix in group_order:
        ordered.extend([c for c in out.columns if c.startswith(prefix)])

    remaining = [c for c in out.columns if c not in target and c not in ordered]
    out = out[target + ordered + remaining]

    # Drop rows with NaNs from lags/rollings
    out = out.dropna()

    return out


In [13]:
# df_features_test = build_feature_set(df_test, lags=(1, 2, 4, 12, 24))
# df_features_train = build_feature_set(df_train, lags=(1, 2, 4, 12, 24))
# df_features_test.to_csv("../clean_data/test_data_features.csv")
# df_features_train.to_csv("../clean_data/train_data_features.csv")

In [14]:
df_features_train_24hr = build_feature_set(df_train, lags=(24, 48))
df_features_train_24hr.to_csv("../clean_data/train_data_features_24hr.csv")
df_features_test_24hr = build_feature_set(df_test, lags=(24, 48))
df_features_test_24hr.to_csv("../clean_data/test_data_features_24hr.csv")